# Day 065 — Exercise 2: Test Battery Runner

Day 062 taught `pytest` and `@pytest.mark.parametrize`. For a product launch you also want a **programmatic test runner** — one you can call from a notebook, a CI script, or a health-check endpoint.

`run_test_battery` takes a list of test-case dicts and a TestClient, runs every case, and returns a structured summary — never raising, always collecting all results.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# A minimal test app to run the battery against
def build_test_app():
    app = FastAPI()
    class _R(BaseModel):
        prompt: str = Field(min_length=1)
    @app.get("/health")
    def h(): return {"status": "ok"}
    @app.post("/echo")
    def echo(req: _R): return {"echo": req.prompt}
    return app


## Task

Implement `run_test_battery(client, cases) -> dict`:

- Iterate `cases`: each has `name`, `method`, `path`, `json` (optional), `expected_status`
- Call `client.request(method, path, json=body)` — handle exceptions
- Compare `r.status_code == expected_status` → `passed: bool`
- Return `{passed, failed, total, results: [{name, expected, got, passed}]}`

## Your Implementation

In [ ]:
def run_test_battery(client, cases: list[dict]) -> dict:
    """Run a list of API test cases and return a summary.

    Each case dict:
        name:            str — human-readable test name
        method:          str — 'GET' or 'POST'
        path:            str — URL path
        json:            dict | None — request body (optional)
        expected_status: int — expected HTTP status code

    Returns:
        {
          'passed':  int,
          'failed':  int,
          'total':   int,
          'results': list[{name, expected, got, passed: bool}],
        }

    Never raises — catch all exceptions and mark as failed.
    """
    # TODO: iterate cases, call client.request(method, path, json=...),
    #       compare status_code, collect results
    raise NotImplementedError


In [ ]:
def run_test_battery(client, cases: list[dict]) -> dict:
    results = []
    for case in cases:
        name     = case["name"]
        method   = case["method"]
        path     = case["path"]
        body     = case.get("json")
        expected = case["expected_status"]
        try:
            r   = client.request(method, path, json=body)
            got = r.status_code
        except Exception as e:
            got = -1
            name = f"{name} [ERROR: {e}]"
        passed = (got == expected)
        results.append({"name": name, "expected": expected,
                        "got": got, "passed": passed})
    passed_n = sum(1 for r in results if r["passed"])
    return {"passed": passed_n, "failed": len(results) - passed_n,
            "total": len(results), "results": results}


## Automated checks

In [ ]:
score, total = 0, 5
try:
    app = build_test_app()
    c   = TestClient(app, raise_server_exceptions=False)

    cases = [
        {"name": "health ok",      "method": "GET",  "path": "/health",
         "json": None,             "expected_status": 200},
        {"name": "echo ok",        "method": "POST", "path": "/echo",
         "json": {"prompt": "hi"}, "expected_status": 200},
        {"name": "echo empty",     "method": "POST", "path": "/echo",
         "json": {"prompt": ""},   "expected_status": 422},
        {"name": "not found",      "method": "GET",  "path": "/missing",
         "json": None,             "expected_status": 404},
        {"name": "intentional fail","method": "GET",  "path": "/health",
         "json": None,             "expected_status": 999},   # wrong expected
    ]

    result = run_test_battery(c, cases)

    # has required keys
    for k in ("passed", "failed", "total", "results"):
        assert k in result, f"Missing key: {k}"
    score += 1; print("\u2705 result has all required keys")

    # total = len(cases)
    assert result["total"] == len(cases)
    score += 1; print("\u2705 total == len(cases)")

    # first 4 should pass, last one fails (expected=999)
    assert result["passed"] == 4
    assert result["failed"] == 1
    score += 1; print("\u2705 passed=4, failed=1 (correct pass/fail split)")

    # results list has one entry per case
    assert len(result["results"]) == len(cases)
    score += 1; print("\u2705 results list has one entry per case")

    # each result has name, expected, got, passed
    for r in result["results"]:
        assert all(k in r for k in ("name", "expected", "got", "passed"))
    score += 1; print("\u2705 each result entry has name/expected/got/passed")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def run_test_battery(client, cases: list[dict]) -> dict:
    results = []
    for case in cases:
        name     = case["name"]
        method   = case["method"]
        path     = case["path"]
        body     = case.get("json")
        expected = case["expected_status"]
        try:
            r   = client.request(method, path, json=body)
            got = r.status_code
        except Exception as e:
            got = -1
            name = f"{name} [ERROR: {e}]"
        passed = (got == expected)
        results.append({"name": name, "expected": expected,
                        "got": got, "passed": passed})
    passed_n = sum(1 for r in results if r["passed"])
    return {"passed": passed_n, "failed": len(results) - passed_n,
            "total": len(results), "results": results}
```

**Never raise in a test runner**: wrapping in try/except means a network error or unexpected exception marks the test failed but lets all other tests run. One crash shouldn't blank out 20 results. `got = -1` for exceptions makes it visually obvious in the results dict.

</details>